In [1]:
# !pip install h5py
import h5py

In [2]:
import numpy as np
import pandas as pd

In [3]:
# Open the file (read-only mode)
def read_as_df(file_obj, key):
    # output dictionary
    out_dict = {}
    
    # references
    refs = file_obj[file_obj[key][()][0]][:]
    counter = 0
    for r in refs:
        scenario_num = file_obj[file_obj[r][()][0]][()]
        out_dict[f"{scenario_num}.{key}"] = np.squeeze(file_obj[file_obj[r][()][1]][:])
        counter += 1

    # return dataframe
    return pd.DataFrame(out_dict)

In [4]:
# with h5py.File("../../data/DataExchangeMilan_Project2.2a/solution/CCD18-3/COTWDPGE_Sol.jld2", "r") as f:
    # refs = f[f["PowerBase"][()][0]][()]
    # sel_indx = 99
    # scaler values: PowerBase, Wrate
    # print(f[f[f["S21"][()][0]][:][0]][()])
    # print(f[f[f[f["S21"][()][0]][0]][()][1]][:])
    # print(f[f[refs[sel_indx]][()][0]][()], f[f[refs[sel_indx]][()][1]][:])
    # read_as_df(f, "PowerBase")

In [5]:
with h5py.File("../../data/DataExchangeMilan_Project2.2a/solution/CCD18-3/COTWDPGE_Sol.jld2", "r") as f:
    # Print all keys (groups/datasets)
    print("Keys: ", list(f.keys()))
    
    # Explore the structure recursively
    def explore(name, obj):
        print(f"{name} - {'Group' if isinstance(obj, h5py.Group) else 'Dataset'}")

    # f.visititems(explore)

    list_of_df96s = []
    list_of_df24s = []
    scaler_dict = {}
    
    # for each key
    for k in f.keys():
        # unable to read keys
        if k in ["PowerBase", "S1", "S2", "S21", "Wrate", "_types"]:
            if k in ["PowerBase", "Wrate", "S1", "S2"]:
                scaler_dict[k] = f[k][()]
            elif k in ["S21"]:
                scaler_dict[k] = read_as_df(f, k)
        elif k in ["pWDS", "pWS", "WS"]:
            list_of_df24s.append(read_as_df(f, k))
        # read as dataframe
        else:
            list_of_df96s.append(read_as_df(f, k))

    # print(f["v1"][:])
    
    # print(f[f[f[ref[0]][:][0]][()][1]][:])
    
    # for i, ref in enumerate(f["v1"][:]):
    #     dset = f[ref]
    #     print(dset)

Keys:  ['ChS', 'DisS', 'PowerBase', 'S1', 'S2', 'S21', 'SCS', 'WPQ', 'WS', 'WSQ', 'Wrate', '_types', 'kBS', 'kWS', 'lam_DAQ', 'lam_RT', 'pRBDS', 'pRBUS', 'pRWDS', 'pRWUS', 'pWDS', 'pWDSQ', 'pWRS', 'pWS', 'pWSQ', 'v1', 'v2']


In [6]:
scaler_dict

{'PowerBase': np.int64(1810),
 'S1': array([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
        19, 20, 21]),
 'S2': array([ 22,  23,  24,  25,  26,  27,  28,  29,  30,  31,  32,  33,  34,
         35,  36,  37,  38,  39,  40,  41,  42,  43,  44,  45,  46,  47,
         48,  49,  50,  51,  52,  53,  54,  55,  56,  57,  58,  59,  60,
         61,  62,  63,  64,  65,  66,  67,  68,  69,  70,  71,  72,  73,
         74,  75,  76,  77,  78,  79,  80,  81,  82,  83,  84,  85,  86,
         87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,  98,  99,
        100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112,
        113, 114, 115, 116, 117, 118, 119, 120, 121]),
 'S21':    5.S21  16.S21  20.S21  12.S21  8.S21  17.S21  19.S21  6.S21  11.S21  9.S21  \
 0     37      92     112      72     52      97     107     42      67     57   
 1     38      93     113      73     53      98     108     43      68     58   
 2     39      94     114      74     54 

In [7]:
df96 = pd.concat(list_of_df96s, axis=1)
df96.columns = pd.MultiIndex.from_tuples(
    [tuple(col.split('.')) for col in df96.columns],
    names=['Scenario', 'Parameter']
)
df96

Scenario,56,35,55,110,114,60,30,32,67,45,...,89,80,96,51,33,40,48,113,65,97
Parameter,ChS,ChS,ChS,ChS,ChS,ChS,ChS,ChS,ChS,ChS,...,v2,v2,v2,v2,v2,v2,v2,v2,v2,v2
0,0.001214,0.001864,0.000996,0.049615,0.001001,0.000843,0.001503,0.004127,0.002486,0.000772,...,508.359355,506.942406,501.782684,503.188085,501.780423,505.928251,501.780611,501.783011,501.782046,508.103074
1,0.001276,0.002220,0.001058,0.067645,0.001065,0.000925,0.001627,0.004198,0.002643,0.000812,...,508.358854,506.942213,501.782662,503.187987,501.780360,505.928192,501.780567,501.782996,501.782016,508.100672
2,0.001346,0.002751,0.001128,0.108324,0.001140,0.001025,0.001775,0.004272,0.002822,0.000858,...,508.358340,506.942016,501.782639,503.187887,501.780296,505.928132,501.780521,501.782981,501.781985,508.098183
3,0.001424,0.003620,0.001208,0.305648,0.001226,0.001149,0.001952,0.004348,0.003028,0.000909,...,508.357810,506.941814,501.782617,503.187784,501.780230,505.928071,501.780475,501.782966,501.781953,508.095602
4,0.001516,0.005285,0.001299,24.500466,0.001326,0.001079,0.002172,0.003901,0.003261,0.000966,...,507.983720,506.828037,501.782679,503.307067,501.780364,505.894387,501.780432,501.782880,501.781902,508.188583
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,38.077140,0.001023,17.122021,27.151022,0.000728,0.000286,0.000717,0.005390,24.403480,0.000898,...,504.201136,504.577342,501.783005,502.875243,501.781023,504.213330,501.781406,501.783094,501.782129,504.350657
92,52.036581,0.001350,36.271573,53.931781,0.000780,0.000297,0.000762,27.123905,30.365095,0.000922,...,506.232257,504.296877,501.782890,503.365776,501.780734,504.236036,501.781270,501.783249,501.782066,503.647639
93,52.036817,0.001146,36.269297,53.934733,0.000717,0.000263,0.000704,27.123350,30.363298,0.000828,...,506.232421,504.296882,501.782907,503.365782,501.780737,504.236063,501.781276,501.783284,501.782078,503.650483


In [8]:
# accessing a scenario
df96.loc[:, ('56', slice(None))]

Scenario          56                                                        \
Parameter        ChS      DisS         SCS         WPQ       WSQ       kBS   
0           0.001214  0.000383  108.601942    0.000000  2.778696  2.999842   
1           0.001276  0.000377  108.602129    0.000000  2.778696  2.999836   
2           0.001346  0.000371  108.602334    0.000000  2.778696  2.999830   
3           0.001424  0.000365  108.602557    0.000000  2.778696  2.999824   
4           0.001516  0.000359  108.602804   23.256503  3.363003  2.999819   
..               ...       ...         ...         ...       ...       ...   
91         38.077140  0.000260  150.360744  472.651282  6.799257  2.999605   
92         52.036581  0.000227  162.199004  518.832760  6.997463  2.999417   
93         52.036817  0.000229  174.037317  518.832760  6.997463  2.998887   
94         52.043469  0.000231  185.877142  518.832760  6.997463  2.989788   
95         53.633475  0.000256  198.078688  518.832760  6.997463  0.862682   

Scenario                                                                    \
Parameter       kWS        lam_RT     pRBDS      pRBUS     pRWDS     pRWUS   
0          0.000000  38873.373507  2.262381  22.623811  0.000000  0.000000   
1          0.000000  38328.268884  2.262377  22.623767  0.000000  0.000000   
2          0.000000  37783.164261  2.262372  22.623720  0.000000  0.000000   
3          0.000000  37238.059638  2.262367  22.623670  0.000000  0.000000   
4          0.025756  36692.955015  2.262364  22.623636  0.019424  0.191087   
..              ...           ...       ...        ...       ...       ...   
91         0.522308  43284.757149  2.262202  22.622018  0.393907  3.875088   
92         0.573342  40990.881041  2.262061  22.620606  0.432395  4.253715   
93         0.573340  41520.257924  2.261660  22.616604  0.432394  4.253706   
94         0.573340  42049.634808  2.254798  22.547984  0.432394  4.253702   
95         0.573339  42579.011691  0.650606   6.506063  0.432393  4.253698   

Scenario                                                    
Parameter         pWRS        pWSQ          v1          v2  
0         -1114.123994    0.000000  482.224793  482.224789  
1         -1114.124065    0.000000  482.225169  482.225165  
2         -1114.124144    0.000000  482.225492  482.225487  
3         -1114.124230    0.000000  482.225780  482.225776  
4         -1083.899806   23.065016  493.586472  493.477397  
..                 ...         ...         ...         ...  
91         -940.420097  468.715054  509.961313  507.815916  
92         -886.375169  514.511622  509.962495  507.607483  
93         -886.375396  514.511622  509.962966  507.607957  
94         -886.382038  514.511622  509.963471  507.608464  
95         -887.971964  514.511622  509.967484  507.612495  

[96 rows x 16 columns]

In [9]:
df24 = pd.concat(list_of_df24s, axis=1)
df24.columns = pd.MultiIndex.from_tuples(
    [tuple(col.split('.')) for col in df24.columns],
    names=['Scenario', 'Parameter']
)
df24

Scenario,56,35,55,110,114,60,30,32,67,45,...,89,80,96,51,33,40,48,113,65,97
Parameter,WS,WS,WS,WS,WS,WS,WS,WS,WS,WS,...,pWS,pWS,pWS,pWS,pWS,pWS,pWS,pWS,pWS,pWS
0,2.778696,5.259182,7.151340,6.663712,2.853987,8.560501,3.926177,23.206838,2.802229,15.031624,...,350.834072,664.498816,1794.918805,1486.055776,1794.917605,888.708941,1794.917705,1794.918999,1794.918503,385.131024
1,3.363003,5.202561,6.872986,5.950005,3.133387,7.887151,3.632909,23.111289,3.330173,14.670545,...,435.370088,689.590926,1794.918813,1460.153476,1794.917603,896.071108,1794.917634,1794.918936,1794.918446,359.546459
2,3.562975,5.000554,6.418881,5.023423,3.466980,7.320037,3.477984,22.467332,3.563793,14.099668,...,471.547976,594.587111,1794.918760,1337.258761,1794.917469,809.310531,1794.917686,1794.918881,1794.918277,343.507581
3,3.479494,5.146632,5.981765,4.595384,3.882581,6.869727,3.564644,22.244075,3.843049,13.852415,...,386.703836,579.318797,1794.918721,1078.784620,1794.917483,807.669632,1794.917912,1794.918862,1794.918221,310.778728
4,3.626147,5.255451,6.165515,4.922666,4.189336,6.517185,3.917294,21.184320,4.264484,13.932399,...,323.177197,623.520805,1794.918819,1056.989219,1794.917642,845.980450,1794.918424,1794.918923,1794.918249,230.629557
5,3.994235,5.187725,6.399563,4.056707,4.213485,6.219640,3.827852,19.945576,4.806050,13.315962,...,247.772036,616.397041,1794.918975,1103.719830,1794.917529,871.723382,1794.918404,1794.918999,1794.918335,299.101231
6,4.482606,5.290227,6.650397,2.905216,4.238918,6.306684,3.736930,19.278687,5.485692,13.096001,...,268.969325,524.108619,1794.919008,1094.934963,1794.917108,895.469531,1794.917934,1794.919056,1794.918466,390.493678
7,4.369471,5.104033,7.182092,2.652289,4.586662,6.338177,3.883066,18.997124,5.499126,13.206703,...,313.711698,454.627816,1794.918807,1028.790625,1794.916717,894.549376,1794.917234,1794.918963,1794.918275,504.820401
8,4.711400,5.495798,7.992253,3.466721,4.966180,5.938256,4.090613,19.297063,5.882396,13.259421,...,272.322344,559.103970,1794.918550,871.780713,1794.916634,949.990020,1794.916787,1794.918870,1794.918244,537.354832


In [10]:
# accessing a scenario
df24.loc[:, ('97', slice(None))]

Scenario         97             
Parameter        WS          pWS
0          6.405215   385.131024
1          6.274580   359.546459
2          6.189874   343.507581
3          6.009535   310.778728
4          5.515301   230.629557
5          5.942521   299.101231
6          6.431942   390.493678
7          6.956440   504.820401
8          7.092883   537.354832
9          6.912918   494.661355
10         7.131795   546.827856
11         7.320943   594.188425
12         7.777112   717.561363
13         8.550888   957.715759
14         8.747920  1025.336004
15         8.785417  1038.512235
16         9.631011  1362.359083
17         9.929919  1489.477416
18         9.838777  1449.998568
19         9.797013  1432.120511
20         9.870897  1463.839878
21         9.462836  1293.794224
22         9.305795  1231.656495
23         9.682939  1383.958166